In [2]:
import numpy as np
from src.evaluation.metrics import ClassificationMetrics
# ---------- Dữ liệu ví dụ tính tay: labels = [1, 2, 3] ----------
Y_TRUE = [1, 1, 1, 2, 2, 3, 3, 3]
Y_PRED = [1, 2, 1, 2, 3, 3, 3, 1]
LABELS = [1, 2, 3]


def test_confusion_matrix():
    m = ClassificationMetrics(Y_TRUE, Y_PRED, labels=LABELS)
    expected = np.array([
        [2, 1, 0],
        [0, 1, 1],
        [1, 0, 2],
    ])
    assert np.array_equal(m.cm, expected), f"cm sai:\n{m.cm}"


def test_row_is_true_col_is_pred():
    m = ClassificationMetrics([0, 1], [1, 1], labels=[0, 1])
    assert m.cm[0, 1] == 1
    assert m.cm[1, 0] == 0


def test_labels_inferred_from_both_arrays():
    m = ClassificationMetrics([0, 0, 1], [0, 2, 1])
    assert np.array_equal(m.labels, [0, 1, 2]), f"labels sai: {m.labels}"


def test_custom_label_order():
    m = ClassificationMetrics([0, 0, 1], [0, 1, 1], labels=[1, 0])
    assert np.array_equal(m.labels, [1, 0])
    assert np.array_equal(m.cm, [[1, 0], [1, 1]])


def test_per_class_metrics():
    r = ClassificationMetrics(Y_TRUE, Y_PRED, labels=LABELS).evaluate()
    assert np.allclose(r["precision"], [2/3, 1/2, 2/3])
    assert np.allclose(r["recall"], [2/3, 1/2, 2/3])
    assert np.allclose(r["f1"], [2/3, 1/2, 2/3])
    assert np.isclose(r["accuracy"], 5/8)
    assert np.isclose(r["macro_f1"], (2/3 + 1/2 + 2/3) / 3)


def test_perfect_prediction():
    y = [0, 1, 2, 1, 0]
    r = ClassificationMetrics(y, y).evaluate()
    assert np.allclose(r["precision"], 1.0)
    assert np.allclose(r["recall"], 1.0)
    assert np.allclose(r["f1"], 1.0)
    assert r["accuracy"] == 1.0


def test_never_predicted_class_gives_zero():
    # Lớp 1 không bao giờ được dự đoán: tp + fp = 0, không được báo lỗi
    m = ClassificationMetrics([0, 1, 1], [0, 0, 0], labels=[0, 1])
    r = m.evaluate()
    assert r["precision"][1] == 0.0
    assert r["recall"][1] == 0.0


def test_absent_class_gives_zero():
    # Lớp 2 không có ở đâu cả
    m = ClassificationMetrics([0, 0, 1], [0, 1, 1], labels=[0, 1, 2])
    r = m.evaluate()
    assert r["precision"][2] == 0.0
    assert r["recall"][2] == 0.0
    assert r["f1"][2] == 0.0


def test_invalid_inputs():
    bad_cases = [
        ([[0, 1]], [[0, 1]], None),      # không phải 1D
        ([0, 1], [0], None),             # lệch độ dài
        ([], [], None),                  # rỗng
        ([0, 1, 2], [0, 1, 2], [0, 1]),  # labels thiếu lớp 2
        ([0, 1], [0, 1], [0, 0, 1]),     # labels trùng
    ]
    for y_true, y_pred, labels in bad_cases:
        try:
            ClassificationMetrics(y_true, y_pred, labels=labels)
        except ValueError:
            continue
        raise AssertionError(f"Không báo lỗi với: {y_true}, {y_pred}, {labels}")


def test_matches_sklearn():
    from sklearn.metrics import precision_score, recall_score, f1_score
    rng = np.random.default_rng(0)
    y_true = rng.integers(0, 4, 200)
    y_pred = rng.integers(0, 4, 200)
    labels = [0, 1, 2, 3]
    r = ClassificationMetrics(y_true, y_pred, labels=labels).evaluate()
    kw = dict(average=None, labels=labels, zero_division=0)
    assert np.allclose(r["precision"], precision_score(y_true, y_pred, **kw))
    assert np.allclose(r["recall"], recall_score(y_true, y_pred, **kw))
    assert np.allclose(r["f1"], f1_score(y_true, y_pred, **kw))


# ---------- Chạy tất cả ----------
tests = [
    test_confusion_matrix,
    test_row_is_true_col_is_pred,
    test_labels_inferred_from_both_arrays,
    test_custom_label_order,
    test_per_class_metrics,
    test_perfect_prediction,
    test_never_predicted_class_gives_zero,
    test_absent_class_gives_zero,
    test_invalid_inputs,
    test_matches_sklearn,
]

passed = 0
for t in tests:
    try:
        t()
        print(f"PASS  {t.__name__}")
        passed += 1
    except AssertionError as e:
        print(f"FAIL  {t.__name__}: {e}")
    except Exception as e:
        print(f"ERROR {t.__name__}: {type(e).__name__}: {e}")

print(f"\n{passed}/{len(tests)} test pass")

PASS  test_confusion_matrix
PASS  test_row_is_true_col_is_pred
PASS  test_labels_inferred_from_both_arrays
PASS  test_custom_label_order
PASS  test_per_class_metrics
PASS  test_perfect_prediction
PASS  test_never_predicted_class_gives_zero
PASS  test_absent_class_gives_zero
PASS  test_invalid_inputs
PASS  test_matches_sklearn

10/10 test pass
